# Clasificación de imágenes con CNN — CIFAR-10

Notebook de exploración, entrenamiento y análisis para el proyecto de
clasificación de imágenes con Redes Neuronales Convolucionales sobre
**CIFAR-10**.

Este notebook cubre:
- Parte 1: Exploración del dataset
- Parte 2: Comprender las convoluciones
- Parte 3-6: Construcción y entrenamiento de la CNN (usando `src/`)
- Parte 7-14: Visualización, evaluación, matriz de confusión, feature maps, inferencia
- Parte 13: Los 4 experimentos obligatorios (llamando a `src/train.py` y `src/evaluate.py`)

> Nota: la lógica reutilizable vive en `src/model.py`, `src/utils.py`,
> `src/train.py` y `src/evaluate.py`. Este notebook los importa para
> mantener el código organizado (buena práctica de ingeniería de software).


In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt
import torch

from model import build_model, CIFAR10_CLASSES
from utils import get_dataloaders, get_device, denormalize, set_seed, count_parameters
from evaluate import evaluate_model, plot_confusion_matrix, plot_predictions, analyze_misclassified, plot_feature_maps, predict_image

set_seed(42)
device = get_device()
device = torch.device("cpu")  # Forzado a CPU para evitar mezclas MPS/CPU
print("Device:", device)


## Parte 1 — Exploración del dataset

### Paso 1. Descargar CIFAR-10 y crear DataLoaders

In [ ]:
train_loader, test_loader = get_dataloaders(data_root="../data", batch_size=64, augment=False)
train_dataset = train_loader.dataset
test_dataset = test_loader.dataset

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))


### Paso 2. Inspeccionar dimensiones

In [ ]:
image, label = train_dataset[0]
print("Shape del tensor:", image.shape)  # Channels x Height x Width
print("Label numerico:", label)
print("Nombre de la clase:", CIFAR10_CLASSES[label])


**Explicación:** el tensor tiene shape `(3, 32, 32)`, es decir
`Channels x Height x Width` (convención de PyTorch, distinta a la de
NumPy/PIL que usan `Height x Width x Channels`).

### Paso 3. Visualizar imágenes (al menos 10)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    img, lbl = train_dataset[i]
    ax.imshow(denormalize(img))
    ax.set_title(CIFAR10_CLASSES[lbl])
    ax.axis("off")
plt.tight_layout()
plt.show()


### Paso 4. Analizar las clases

Responder:

1. ¿Cuántas clases existen? → 10 clases (ver `CIFAR10_CLASSES`).
2. ¿Cuántas imágenes tiene training? → ver celda de abajo.
3. ¿Cuántas imágenes tiene test? → ver celda de abajo.
4. ¿El dataset está balanceado? → CIFAR-10 tiene exactamente 5,000
   imágenes por clase en train y 1,000 en test (verificar abajo).
5. ¿Qué clases parecen más fáciles de distinguir? *(completar tras ver
   la matriz de confusión, Parte 9)*
6. ¿Qué clases podrían confundirse? *(completar tras ver la matriz de
   confusión — animales similares como gato/perro o vehículos como
   automobile/truck son candidatos naturales)*

In [ ]:
import collections

train_labels = [label for _, label in train_dataset]
counts = collections.Counter(train_labels)
for idx, name in enumerate(CIFAR10_CLASSES):
    print(f"{name:12s}: {counts[idx]} imagenes")


## Parte 2 — Comprender las convoluciones

Experimentamos con una única capa `Conv2d` antes de construir la CNN completa.

In [ ]:
import torch.nn as nn

conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)

image, label = train_dataset[0]
input_batch = image.unsqueeze(0)  # (1, 3, 32, 32)

with torch.no_grad():
    output = conv(input_batch)

print("Dimension de entrada :", input_batch.shape)
print("Dimension de salida  :", output.shape)


**Respuestas:**

1. `in_channels=3`: la imagen de entrada tiene 3 canales de color (RGB).
2. `out_channels=16`: la capa aprende 16 filtros distintos, produciendo
   16 feature maps de salida.
3. `kernel_size=3`: cada filtro es una ventana de 3x3 píxeles que se
   desliza sobre la imagen.
4. `padding=1`: agrega un borde de 1 píxel de ceros alrededor de la
   imagen para que la convolución no reduzca el tamaño espacial
   (con kernel 3x3 y padding 1, la salida mantiene 32x32).
5. Se obtienen 16 feature maps porque `out_channels=16` define
   explícitamente 16 filtros/kernels distintos, cada uno generando su
   propio mapa de activación al convolucionar sobre los 3 canales de
   entrada.

## Parte 3-6 — Construcción y entrenamiento de la CNN

La arquitectura vive en `src/model.py` (clase `CIFAR10CNN`). Aquí
construimos el modelo básico y verificamos el flujo de dimensiones
(Parte 4) antes de entrenar.

In [ ]:
model = build_model("basic").to(device)
print(model)
print("\nParametros entrenables:", count_parameters(model))


In [ ]:
# Verificacion del flujo de dimensiones (Parte 4)
x = torch.randn(1, 3, 32, 32).to(device)
h = x
for layer in model.features:
    h = layer(h)
    print(f"{layer.__class__.__name__:15s} -> {tuple(h.shape)}")


**Respuestas (Parte 4):**

1. El tamaño espacial disminuye por el `MaxPool2d(2,2)`, que reduce a
   la mitad alto y ancho en cada bloque (32→16→8→4), quedándose con el
   valor máximo de cada ventana 2x2 y descartando resolución espacial
   a cambio de invariancia a pequeñas traslaciones.
2. El número de canales aumenta (32→64→128) porque cada bloque
   convolucional adicional puede combinar los feature maps del bloque
   anterior en representaciones más abstractas; más canales = más
   "conceptos" distintos que la red puede representar en cada posición.
3. Cada filtro aprende a detectar un patrón visual local: en capas
   tempranas bordes, texturas y colores; en capas profundas, combinaciones
   de esos patrones que se acercan a partes de objetos.
4. `Flatten` convierte el tensor 3D `(canales, alto, ancho)` en un
   vector 1D para poder alimentarlo a las capas densas (`Linear`), que
   solo aceptan vectores.

Para entrenar los 4 experimentos obligatorios (Parte 13), se recomienda
correrlos desde terminal (más estable para runs largos), y luego cargar
los resultados aquí para el análisis:

```bash
python ../src/train.py --experiment basic
python ../src/train.py --experiment deep
python ../src/train.py --experiment dropout
python ../src/train.py --experiment augmentation
```

También se puede entrenar directamente desde el notebook:

In [ ]:
# Entrenamiento del experimento basico directamente desde el notebook
# (usar el script train.py es mas comodo para runs largos)

import subprocess
result = subprocess.run(
    ["python", "../src/train.py", "--experiment", "basic", "--epochs", "10"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)


## Parte 7 — Visualización del entrenamiento

Las curvas se generan automáticamente en `train.py` y se guardan en
`results/{experimento}_training_loss.png` y
`results/{experimento}_training_accuracy.png`.

In [ ]:
from PIL import Image

loss_img = Image.open("../results/basic_training_loss.png")
acc_img = Image.open("../results/basic_training_accuracy.png")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(loss_img); axes[0].axis("off")
axes[1].imshow(acc_img); axes[1].axis("off")
plt.show()


**Responder tras ver las curvas:**

1. ¿Disminuye el loss? *(completar con la observación real)*
2. ¿Aumenta el accuracy? *(completar)*
3. ¿Parece que el modelo aprende? *(completar)*
4. ¿Hay señales de inestabilidad (oscilaciones fuertes, loss que sube)? *(completar)*

## Parte 8 — Evaluación sobre el conjunto de test

Se calculan accuracy, precision, recall y F1-score **únicamente sobre
test** (nunca sobre train, para no sesgar la medición de generalización).

In [ ]:
model.load_state_dict(torch.load("../models/basic_cifar10_cnn.pth", map_location=device))
metrics = evaluate_model(model, test_loader, device)


## Parte 9 — Matriz de confusión

In [ ]:
cm = plot_confusion_matrix(metrics["y_true"], metrics["y_pred"], results_dir="../results", prefix="basic_")

from PIL import Image
plt.figure(figsize=(8,7))
plt.imshow(Image.open("../results/basic_confusion_matrix.png"))
plt.axis("off")
plt.show()


**Analizar (completar con la matriz obtenida):**

1. ¿Qué clase se identifica mejor?
2. ¿Qué clase se identifica peor?
3. ¿Qué clases se confunden entre sí?
4. ¿Por qué podrían confundirse? (similitud visual, forma, fondo, pose)

## Parte 10 — Visualización de predicciones (5 correctas + 5 incorrectas)

In [ ]:
plot_predictions(model, test_dataset, device, results_dir="../results", prefix="basic_")

plt.figure(figsize=(12,5))
plt.imshow(Image.open("../results/basic_predictions.png"))
plt.axis("off")
plt.show()


## Parte 11 — Análisis de errores

Se seleccionan 10 imágenes mal clasificadas con su confianza (softmax).

In [ ]:
errors = analyze_misclassified(model, test_dataset, device, n=10)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, err in zip(axes.flatten(), errors):
    ax.imshow(denormalize(err["image"]))
    ax.set_title(f"Real: {err['true_label']}\nPred: {err['pred_label']} ({err['confidence']*100:.0f}%)", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


**Responder por cada imagen (o en general para el conjunto de errores):**

1. ¿La imagen es ambigua?
2. ¿El objeto ocupa poco espacio en el cuadro?
3. ¿El fondo puede estar confundiendo a la CNN?
4. ¿La baja resolución (32x32) puede explicar el error?
5. ¿La clase predicha tiene similitudes visuales con la clase correcta?

## Parte 12 — Visualización de Feature Maps

In [ ]:
sample_image, sample_label = test_dataset[0]
plot_feature_maps(model, sample_image, results_dir="../results", prefix="basic_", n_maps=16)

plt.figure(figsize=(8,8))
plt.imshow(Image.open("../results/basic_feature_maps.png"))
plt.axis("off")
plt.show()


**Explicar:**

- Los mapas son diferentes porque cada filtro tiene pesos distintos
  (inicializados al azar y luego ajustados por backpropagation), por lo
  que cada uno "busca" un patrón distinto.
- En la primera capa, típicamente detectan bordes, cambios de color y
  texturas simples.
- No todos los filtros responden igual a la misma imagen porque cada
  filtro está especializado en un patrón particular; si ese patrón no
  aparece en la imagen, el filtro produce activaciones bajas (mapa casi
  plano/oscuro).

## Parte 13 — Experimentos obligatorios

Correr los 4 experimentos (ya sea desde esta celda o desde terminal
usando `train.py --experiment ...` y luego `evaluate.py --experiment ...`).

In [ ]:
experiments = ["basic", "deep", "dropout", "augmentation"]
results_summary = {}

for exp in experiments:
    print(f"\n=== Entrenando experimento: {exp} ===")
    subprocess.run(["python", "../src/train.py", "--experiment", exp, "--epochs", "10"])


In [ ]:
for exp in experiments:
    print(f"\n=== Evaluando experimento: {exp} ===")
    m = build_model(exp).to(device)
    m.load_state_dict(torch.load(f"../models/{exp}_cifar10_cnn.pth", map_location=device))
    metrics_exp = evaluate_model(m, test_loader, device)
    results_summary[exp] = metrics_exp


## Parte 19 — Tabla comparativa obligatoria

Completar con los valores obtenidos en `results_summary`:

| Modelo | Train Accuracy | Test Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| CNN básica | | | | | |
| CNN profunda | | | | | |
| CNN + Dropout | | | | | |
| CNN + Data Augmentation | | | | | |

**Seleccionar el mejor modelo y justificar** *(completar tras comparar)*.

In [ ]:
for exp, m in results_summary.items():
    print(f"{exp:14s} | Acc: {m['accuracy']*100:5.2f}% | Prec: {m['precision']*100:5.2f}% | Rec: {m['recall']*100:5.2f}% | F1: {m['f1']*100:5.2f}%")


## Parte 21 — Inferencia con `predict_image`

In [ ]:
sample_image, sample_label = test_dataset[5]
result = predict_image(sample_image, model, device=device, top_k=3)

print("Clase real     :", CIFAR10_CLASSES[sample_label])
print("Clase predicha :", result["predicted_class"])
print("Probabilidad   :", f"{result['probability']*100:.2f}%")
print("Top-3          :", result["top_k"])


## Parte 22 — Preguntas conceptuales obligatorias

*(Completar cada respuesta con tus propias palabras; se listan los
puntos clave a cubrir como guía de estudio.)*

### CNN
1. ¿Por qué una CNN es más adecuada que una ANN tradicional para imágenes?
2. ¿Qué es una convolución?
3. ¿Qué es un kernel?
4. ¿Qué aprende un filtro?
5. ¿Qué es un feature map?
6. ¿Qué función cumple ReLU?
7. ¿Qué función cumple MaxPooling?
8. ¿Qué significa receptive field?

### Entrenamiento
9. ¿Qué función cumple el loss?
10. ¿Qué hace backpropagation?
11. ¿Qué hace Adam?
12. ¿Qué representa el learning rate?
13. ¿Qué significa epoch?
14. ¿Qué significa batch size?

### Generalización
15. ¿Qué es overfitting?
16. ¿Cómo puede detectarse?
17. ¿Cómo puede ayudar Dropout?
18. ¿Cómo puede ayudar Data Augmentation?

## Parte 23 — Pregunta central del proyecto

> Una CNN nunca recibe instrucciones explícitas como "buscar ruedas",
> "detectar orejas" o "identificar alas". ¿Cómo consigue aprender
> representaciones visuales útiles únicamente a partir de imágenes y
> etiquetas?

*(Responder conectando: Imagen → Convoluciones → Feature Maps →
Predicción → Loss → Backpropagation → Gradientes → Actualización de
kernels → Mejores representaciones.)*